In [1]:
import numpy as np 
import pandas as pd

In [6]:
import requests
import pandas as pd

url = "https://en.wikipedia.org/wiki/List_of_Telugu_films_of_2021"  # replace with your link

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
response = requests.get(url, headers=headers)

# Parse the HTML tables
tables = pd.read_html(response.text, header=0)

df1 = tables[2]
df2 = tables[3]
df3 = tables[4]
df4 = tables[5]
df5 = tables[6]


C:\Users\Vishnu\AppData\Local\Temp\ipykernel_2420\1624086127.py:10: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text, header=0)


In [9]:
df=pd.concat([df2,df3,df4,df5],ignore_index=True)

In [10]:
df.head()

,Opening,Opening.1,Title,Director,Cast,Production company,Ref.,Production House
0,J A N U A R Y,9,Krack,Gopichand Malineni,Ravi TejaShruti HaasanVaralaxmi SarathkumarSam...,Saraswathi Films Division,[14][15],NaN
1,J A N U A R Y,12,Mail,Uday Gurrala,Priyadarshi PulikondaHarshith MalgireddyMani A...,Swapna Cinema,[16][17],NaN
2,J A N U A R Y,14,Red,Kishore Tirumala,Ram PothineniNivetha PethurajMalvika SharmaAmr...,Sri Sravanthi Movies,[18][19],NaN
3,J A N U A R Y,14,Alludu Adhurs,Santhosh Srinivas,Bellamkonda SreenivasNabha NateshAnu EmmanuelS...,Sumanth Movie Productions,[20],NaN
4,J A N U A R Y,15,Cycle,Aatla Arjun Reddy,Punarnavi BhupalamMahat RaghavendraSudarshanSw...,Overseas Entertainment Network,[21][22],NaN


In [11]:
df.count()

Opening               114
Opening.1             114
Title                 114
Director              114
Cast                  114
Production company     41
Ref.                  113
Production House       72
dtype: int64

In [13]:
df.drop(['Production House'],axis=1)

,Opening,Opening.1,Title,Director,Cast,Production company,Ref.
0,J A N U A R Y,9,Krack,Gopichand Malineni,Ravi TejaShruti HaasanVaralaxmi SarathkumarSam...,Saraswathi Films Division,[14][15]
1,J A N U A R Y,12,Mail,Uday Gurrala,Priyadarshi PulikondaHarshith MalgireddyMani A...,Swapna Cinema,[16][17]
2,J A N U A R Y,14,Red,Kishore Tirumala,Ram PothineniNivetha PethurajMalvika SharmaAmr...,Sri Sravanthi Movies,[18][19]
3,J A N U A R Y,14,Alludu Adhurs,Santhosh Srinivas,Bellamkonda SreenivasNabha NateshAnu EmmanuelS...,Sumanth Movie Productions,[20]
4,J A N U A R Y,15,Cycle,Aatla Arjun Reddy,Punarnavi BhupalamMahat RaghavendraSudarshanSw...,Overseas Entertainment Network,[21][22]
...,...,...,...,...,...,...,...
109,D E C,24,Shyam Singha Roy,Rahul Sankrityan,NaniSai PallaviKrithi ShettyMadonna SebastianJ...,NaN,[118]
110,D E C,24,WWW: Who Where Why,K. V. Guhan,Adith ArunShivani RajashekarPriyadarshi Puliko...,NaN,[119]
111,D E C,25,Gudaputani,KM Kumar,SaptagiriRaghu KuncheAnanth,NaN,[120]
112,D E C,31,Arjuna Phalguna,Teja Marni,Sree VishnuNareshSivaji RajaSubbarajuDevi Pras...,NaN,[121]


In [14]:
from tmdbv3api import TMDb
import json
import requests
tmdb=TMDb()
tmdb.api_key='08d38c60a56fe3fdf55456105dab7193'

In [16]:
import time
from requests.exceptions import RequestException

# simple cache to avoid duplicate API calls for the same title
_genre_cache = {}

def get_genre_safe(title, max_retries=3, backoff=1.0, pause=0.25):
	# handle missing/empty titles
	if not title or title.lower() in ('nan', 'none', ''):
		return np.nan

	if title in _genre_cache:
		return _genre_cache[title]

	for attempt in range(max_retries):
		try:
			result = tmdb_movie.search(title)
			if not result:
				_genre_cache[title] = np.nan
				return np.nan

			movie_id = result[0].id
			response = requests.get(
				f'https://api.themoviedb.org/3/movie/{movie_id}?api_key={tmdb.api_key}',
				timeout=10
			)
			response.raise_for_status()
			data_json = response.json()

			genres = [g.get('name') for g in data_json.get('genres', []) if g.get('name')]
			value = ' '.join(genres) if genres else np.nan
			_genre_cache[title] = value

			# small pause to reduce likelihood of hitting rate limits / abrupt disconnects
			time.sleep(pause)
			return value

		except RequestException:
			# retry with exponential backoff on network errors
			if attempt < max_retries - 1:
				time.sleep(backoff * (2 ** attempt))
				continue
			_genre_cache[title] = np.nan
			return np.nan
		except Exception:
			# catch-all: return NaN for unexpected failures
			_genre_cache[title] = np.nan
			return np.nan

# apply the safe function to Title column
df['genres'] = df['Title'].fillna('').astype(str).map(lambda x: get_genre_safe(x))
df

,Opening,Opening.1,Title,Director,Cast,Production company,Ref.,Production House,genres
0,J A N U A R Y,9,Krack,Gopichand Malineni,Ravi TejaShruti HaasanVaralaxmi SarathkumarSam...,Saraswathi Films Division,[14][15],NaN,NaN
1,J A N U A R Y,12,Mail,Uday Gurrala,Priyadarshi PulikondaHarshith MalgireddyMani A...,Swapna Cinema,[16][17],NaN,NaN
2,J A N U A R Y,14,Red,Kishore Tirumala,Ram PothineniNivetha PethurajMalvika SharmaAmr...,Sri Sravanthi Movies,[18][19],NaN,NaN
3,J A N U A R Y,14,Alludu Adhurs,Santhosh Srinivas,Bellamkonda SreenivasNabha NateshAnu EmmanuelS...,Sumanth Movie Productions,[20],NaN,NaN
4,J A N U A R Y,15,Cycle,Aatla Arjun Reddy,Punarnavi BhupalamMahat RaghavendraSudarshanSw...,Overseas Entertainment Network,[21][22],NaN,NaN
...,...,...,...,...,...,...,...,...,...
109,D E C,24,Shyam Singha Roy,Rahul Sankrityan,NaniSai PallaviKrithi ShettyMadonna SebastianJ...,NaN,[118],Niharika Entertainment,NaN
110,D E C,24,WWW: Who Where Why,K. V. Guhan,Adith ArunShivani RajashekarPriyadarshi Puliko...,NaN,[119],Ramantra creations,NaN
111,D E C,25,Gudaputani,KM Kumar,SaptagiriRaghu KuncheAnanth,NaN,[120],SRR Productions,NaN
112,D E C,31,Arjuna Phalguna,Teja Marni,Sree VishnuNareshSivaji RajaSubbarajuDevi Pras...,NaN,[121],Matinee Entertainment,NaN
